In [1]:
import numpy as np
import pandas as pd

   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------- ----------- 1.6/2.2 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 2.2/2.2 MB 8.2 MB/s  0:00:00
   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
   ---------- ----------------------------- 2.9/11.1 MB 13.9 MB/s eta 0:00:01
   ------------------- -------------------- 5.5/11.1 MB 13.3 MB/s eta 0:00:01
   --------------------------- ------------ 7.6/11.1 MB 12.2 MB/s eta 0:00:01
   -------------------------------------- - 10.7/11.1 MB 12.9 MB/s eta 0:00:01
   ---------------------------------------- 11.1/11.1 MB 12.6 MB/s  0:00:00

   -------- ------------------------------- 1/5 [wrapt]
  Attempting uninstall: pandas
   -------- ------------------------------- 1/5 [wrapt]
    Found existing installation: pandas 3.0.5
   -------- ------------------------------- 1/5 [wrapt]
   ---------------- ----------------------- 2/5 [pandas]
   ---------------- 

In [5]:
from pykrx import stock
import datetime

# 삼성전자 티커: 005930
ticker = "005930"
today = datetime.datetime.today().strftime("%Y%m%d")
# 최근 7일간의 시세 조회
start_date = (datetime.datetime.today() - datetime.timedelta(days=7)).strftime("%Y%m%d")

print(f"조회 기간: {start_date} ~ {today}")
df = stock.get_market_ohlcv_by_date(start_date, today, ticker)
print(df)

조회 기간: 20260816 ~ 20260823
                시가      고가      저가      종가       거래량       등락률
날짜                                                            
2026-08-18  283000  288000  265000  268500  24464621 -2.185792
2026-08-19  251500  254500  246500  247500  22788552 -7.821229
2026-08-20  257000  273000  252500  271000  26095919  9.494949
2026-08-21  267000  285000  266000  281500  27672192  3.874539


In [ ]:
# 네이버 뉴스 검색 API (https://developers.naver.com/docs/serviceapi/search/news/news.md)
import os
import requests
import pandas as pd
from datetime import datetime
from email.utils import parsedate_to_datetime
from html import unescape

# 환경변수에서 API 키 읽기 — 터미널에서 미리 설정하거나 .env 파일 사용
# 절대 코드에 직접 값을 적지 마세요 (GitHub에 올라가면 키가 노출됩니다)
CLIENT_ID = os.environ["NAVER_CLIENT_ID"]
CLIENT_SECRET = os.environ["NAVER_CLIENT_SECRET"]

url = "https://openapi.naver.com/v1/search/news.json"
headers = {
    "X-Naver-Client-Id": CLIENT_ID,
    "X-Naver-Client-Secret": CLIENT_SECRET,
}

def search_news(query: str, display: int = 20, start: int = 1, sort: str = "date") -> pd.DataFrame:
    """네이버 뉴스 검색 결과를 DataFrame으로 반환. sort: date(최신순) | sim(정확도순)"""
    params = {"query": query, "display": display, "start": start, "sort": sort}
    resp = requests.get(url, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    items = resp.json().get("items", [])
    rows = [
        {
            "title": unescape(item["title"].replace("<b>", "").replace("</b>", "")),
            "description": unescape(item["description"].replace("<b>", "").replace("</b>", "")),
            "link": item["link"],
            "originallink": item["originallink"],
            "pubDate": parsedate_to_datetime(item["pubDate"]),
        }
        for item in items
    ]
    return pd.DataFrame(rows)

# 삼성전자 관련 뉴스 최근 20건
news_df = search_news("트럼프 정부", display=20, sort="date")
print(f"수집된 뉴스 수: {len(news_df)}")
news_df.head(10)


In [ ]:
# Grok(xAI) API 연결 테스트 (https://docs.x.ai/)
import os
import requests as _requests

# 환경변수에서 API 키 읽기 (콘솔: https://console.x.ai)
# 절대 코드에 직접 값을 적지 마세요 (GitHub에 올라가면 키가 노출됩니다)
XAI_API_KEY = os.environ["XAI_API_KEY"]

GROK_URL = "https://api.x.ai/v1/chat/completions"
GROK_MODEL = "grok-4.6"  # 대안: grok-4.5, grok-4-1-fast-non-reasoning 등

def ask_grok(prompt: str, system: str = "You are a helpful assistant.", temperature: float = 0.7) -> str:
    """Grok 챗 컴플리션 호출 후 응답 텍스트 반환"""
    resp = _requests.post(
        GROK_URL,
        headers={
            "Authorization": f"Bearer {XAI_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "model": GROK_MODEL,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": prompt},
            ],
            "temperature": temperature,
        },
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

# 간단 연결 테스트
print(ask_grok("한 문장으로 인사말을 해줘."))


In [ ]:
# 한국거래소(KRX) OpenAPI — 실제 주가 확인 (https://openapi.krx.co.kr)
import requests as _requests2
import pandas as _pd2

# 아래에 KRX 인증키 입력 (발급: openapi.krx.co.kr → 마이페이지 → API 인증키 신청)
# 주의: 키 발급 후 "서비스 이용 → 주식 → 유가증권 일별매매정보"의 API 이용신청도 필요 (승인 약 1일)
KRX_AUTH_KEY = "여기에_KRX_인증키_입력"

KRX_BASE_URL = "https://data-dbg.krx.co.kr/svc/apis"

def get_krx_daily_trade(bas_dd: str, market: str = "stk") -> _pd2.DataFrame:
    """일자별 전종목 매매정보 조회. market: stk(코스피) | ksq(코스닥)"""
    url = f"{KRX_BASE_URL}/sto/{market}_bydd_trd"
    resp = _requests2.get(url, params={"AUTH_KEY": KRX_AUTH_KEY, "basDd": bas_dd}, timeout=30)
    resp.raise_for_status()
    return _pd2.DataFrame(resp.json().get("OutBlock_1", []))

# 최근 영업일 기준 전체 종목 시세 → 삼성전자만 필터링
# (이 API는 날짜 단위로 전 종목을 반환하므로 종목코드 ISU_SRT_CD로 필터링함)
krx_df = get_krx_daily_trade("20260821")
print(f"조회된 종목 수: {len(krx_df)}")
samsung_row = krx_df[krx_df["ISU_SRT_CD"] == "005930"]
samsung_row[["ISU_SRT_CD", "ISU_ABBRV", "TDD_CLSPRC", "CMPPREVDD_PRC", "FLUC_RT", "ACC_TRDVOL"]]
